# TripPulse — Week 7
## Gold Model, KPI Implementation and Reconciliation

**Project:** TripPulse — Urban Mobility Analytics  
**Week:** 7  
**Technology:** Databricks Free Edition | Spark SQL | Delta tables

### Purpose
Build the approved TripPulse Gold layer from **Trusted Silver only**.

This notebook is aligned to the Week-4 Bronze, Week-5 Silver Candidate and Week-6 Data Quality notebooks supplied for TripPulse.

**Gold objects built here**

- `dim_date`
- `dim_zone`
- `dim_driver`
- `fact_trip`
- `fact_payment_attempt`
- `agg_trip_operations_daily`
- `agg_zone_demand_daily`
- `agg_driver_performance_daily`
- `agg_surge_impact_daily`
- `agg_payment_reliability_daily`

The notebook also implements and validates KPI-01 through KPI-10.

> Do not use Bronze, Silver Candidate or Quarantine as Gold inputs. Gold reads the Week-6 Trusted Silver tables only.


## 1. Week-6 handoff used by this notebook

The supplied Week-6 notebook creates these governed inputs:

| Entity | Trusted Silver |
|---|---|
| Zone | `silver_trippulse_zones_trusted` |
| Driver | `silver_trippulse_drivers_trusted` |
| Trip | `silver_trippulse_trips_trusted` |
| Payment | `silver_trippulse_payments_trusted` |

The Week-6 routing rule is:

`Candidate = Trusted Silver + Quarantine`

Gold must consume the trusted side only.


In [0]:
%sql
USE CATALOG `TripPulse`;
USE SCHEMA `default`;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


active_catalog,active_schema
trippulse,default


In [0]:
%sql
SHOW TABLES LIKE 'silver_trippulse_*_trusted';


database,tableName,isTemporary
default,silver_trippulse_drivers_trusted,false
default,silver_trippulse_payments_trusted,false
default,silver_trippulse_trips_trusted,false
default,silver_trippulse_zones_trusted,false


## 2. Reconfirm the Week-6 reconciliation before Gold

This is a gate. If any row is not accounted for, stop and fix Week 6 before building Gold.


In [0]:
%sql
WITH reconciliation AS (
    SELECT
        'zones' AS entity,
        (SELECT COUNT(*) FROM silver_zones_candidate) AS candidate_rows,
        (SELECT COUNT(*) FROM silver_trippulse_zones_trusted) AS trusted_rows,
        (SELECT COUNT(*) FROM quarantine_trippulse_zones) AS quarantine_rows
    UNION ALL
    SELECT
        'drivers',
        (SELECT COUNT(*) FROM silver_drivers_candidate),
        (SELECT COUNT(*) FROM silver_trippulse_drivers_trusted),
        (SELECT COUNT(*) FROM quarantine_trippulse_drivers)
    UNION ALL
    SELECT
        'trips',
        (SELECT COUNT(*) FROM silver_trips_candidate),
        (SELECT COUNT(*) FROM silver_trippulse_trips_trusted),
        (SELECT COUNT(*) FROM quarantine_trippulse_trips)
    UNION ALL
    SELECT
        'payments',
        (SELECT COUNT(*) FROM silver_payments_candidate),
        (SELECT COUNT(*) FROM silver_trippulse_payments_trusted),
        (SELECT COUNT(*) FROM quarantine_trippulse_payments)
)
SELECT *,
       CASE
           WHEN candidate_rows = trusted_rows + quarantine_rows THEN 'PASS'
           ELSE 'CHECK'
       END AS reconciliation_status
FROM reconciliation
ORDER BY entity;


entity,candidate_rows,trusted_rows,quarantine_rows,reconciliation_status
drivers,2800,2793,7,PASS
payments,180315,172286,8029,PASS
trips,250875,241654,9221,PASS
zones,120,120,0,PASS


## 3. Gold object contract

The approved TripPulse Gold model has:

- 3 dimensions
- 2 detailed facts
- 5 batch summaries
- 10 governed KPIs

The exact grains are kept explicit because Trip and PaymentAttempt have different grains.


In [0]:
%sql
SELECT 'dim_date' AS gold_object, 'One calendar date' AS grain
UNION ALL SELECT 'dim_zone', 'One accepted zone'
UNION ALL SELECT 'dim_driver', 'One accepted driver'
UNION ALL SELECT 'fact_trip', 'One trusted trip request'
UNION ALL SELECT 'fact_payment_attempt', 'One trusted payment attempt'
UNION ALL SELECT 'agg_trip_operations_daily', 'One date x service_type'
UNION ALL SELECT 'agg_zone_demand_daily', 'One date x pickup_zone x service_type'
UNION ALL SELECT 'agg_driver_performance_daily', 'One date x driver'
UNION ALL SELECT 'agg_surge_impact_daily', 'One date x pickup_zone x service_type x surge_band'
UNION ALL SELECT 'agg_payment_reliability_daily', 'One date x payment_method';


gold_object,grain
dim_date,One calendar date
dim_zone,One accepted zone
dim_driver,One accepted driver
fact_trip,One trusted trip request
fact_payment_attempt,One trusted payment attempt
agg_trip_operations_daily,One date x service_type
agg_zone_demand_daily,One date x pickup_zone x service_type
agg_driver_performance_daily,One date x driver
agg_surge_impact_daily,One date x pickup_zone x service_type x surge_band
agg_payment_reliability_daily,One date x payment_method


## 4. Build `dim_date`

The date dimension covers the approved batch period and includes 2026-04-01 as the required hand-off/streaming boundary date.

No facts are invented for dates with no activity.


In [0]:
%sql
CREATE OR REPLACE TABLE dim_date
USING DELTA
AS
SELECT
    date_key,
    YEAR(date_key) AS year,
    QUARTER(date_key) AS quarter,
    MONTH(date_key) AS month,
    DATE_FORMAT(date_key, 'MMMM') AS month_name,
    WEEKOFYEAR(date_key) AS week_of_year,
    DATE_FORMAT(date_key, 'EEEE') AS day_of_week,
    CASE WHEN DAYOFWEEK(date_key) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM (
    SELECT EXPLODE(
        SEQUENCE(DATE '2026-01-01', DATE '2026-04-01', INTERVAL 1 DAY)
    ) AS date_key
);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS date_rows,
       MIN(date_key) AS min_date,
       MAX(date_key) AS max_date,
       COUNT(DISTINCT date_key) AS distinct_dates
FROM dim_date;


date_rows,min_date,max_date,distinct_dates
91,2026-01-01,2026-04-01,91


## 5. Build `dim_zone`

Only accepted Week-6 Trusted Silver zone records are used. The table is one row per `zone_id`.


In [0]:
%sql
CREATE OR REPLACE TABLE dim_zone
USING DELTA
AS
SELECT
    zone_id,
    zone_name,
    zone_type,
    city_code,
    demand_band,
    is_active,
    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM silver_trippulse_zones_trusted;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS trusted_zone_rows,
    COUNT(DISTINCT zone_id) AS distinct_zone_ids,
    SUM(CASE WHEN zone_id IS NULL THEN 1 ELSE 0 END) AS null_zone_ids
FROM dim_zone;


trusted_zone_rows,distinct_zone_ids,null_zone_ids
120,120,0


## 6. Build `dim_driver`

`tenure_days` is measured at the approved batch end date, 2026-03-31.


In [0]:
%sql
CREATE OR REPLACE TABLE dim_driver
USING DELTA
AS
SELECT
    driver_id,
    home_zone_id,
    vehicle_type,
    service_type,
    driver_status AS status,
    rating,
    DATEDIFF(DATE '2026-03-31', onboard_date) AS tenure_days,
    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM silver_trippulse_drivers_trusted;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS trusted_driver_rows,
    COUNT(DISTINCT driver_id) AS distinct_driver_ids,
    SUM(CASE WHEN driver_id IS NULL THEN 1 ELSE 0 END) AS null_driver_ids,
    SUM(CASE WHEN home_zone_id IS NULL THEN 1 ELSE 0 END) AS null_home_zone_ids
FROM dim_driver;


trusted_driver_rows,distinct_driver_ids,null_driver_ids,null_home_zone_ids
2793,2793,0,0


## 7. Validate dimensions

These checks prove uniqueness and reference integrity before facts are built.


In [0]:
%sql
SELECT 'dim_date_duplicate_keys' AS check_name, COUNT(*) AS failures
FROM (
    SELECT date_key
    FROM dim_date
    GROUP BY date_key
    HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'dim_zone_duplicate_keys', COUNT(*)
FROM (
    SELECT zone_id
    FROM dim_zone
    GROUP BY zone_id
    HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'dim_driver_duplicate_keys', COUNT(*)
FROM (
    SELECT driver_id
    FROM dim_driver
    GROUP BY driver_id
    HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'dim_driver_unresolved_home_zone', COUNT(*)
FROM dim_driver d
LEFT JOIN dim_zone z ON d.home_zone_id = z.zone_id
WHERE d.home_zone_id IS NOT NULL AND z.zone_id IS NULL;


check_name,failures
dim_date_duplicate_keys,0
dim_zone_duplicate_keys,0
dim_driver_duplicate_keys,0
dim_driver_unresolved_home_zone,0


## 8. Build `fact_trip`

**Grain:** one trusted trip request.

This fact keeps the Trip grain separate from payment attempts. That prevents payment retries from multiplying trip KPIs.

The Week-5 notebook already derived `response_seconds`, `wait_seconds`, `trip_duration_seconds`, `is_completed`, `is_cancelled`, `is_unfulfilled`, `is_surge_trip`, `distance_variance_km` and `fare_variance_inr`.


In [0]:
%sql
CREATE OR REPLACE TABLE fact_trip
USING DELTA
AS
SELECT
    trip_id,
    TO_DATE(request_ts) AS request_date_key,
    driver_id,
    pickup_zone_id,
    dropoff_zone_id,
    service_type,
    trip_status,
    request_ts,
    driver_accept_ts,
    pickup_ts,
    dropoff_ts,
    cancel_ts,

    1 AS trip_count,

    CASE WHEN is_completed = TRUE THEN 1 ELSE 0 END AS completion_flag,
    CASE WHEN is_cancelled = TRUE THEN 1 ELSE 0 END AS cancellation_flag,
    CASE WHEN is_unfulfilled = TRUE THEN 1 ELSE 0 END AS unfulfilled_flag,
    CASE WHEN driver_id IS NOT NULL AND TRIM(driver_id) <> '' THEN 1 ELSE 0 END AS assigned_flag,
    CASE WHEN trip_status = 'cancelled_by_driver' THEN 1 ELSE 0 END AS driver_cancellation_flag,

    response_seconds,
    wait_seconds,
    trip_duration_seconds,

    estimated_distance_km,
    actual_distance_km,
    distance_variance_km,

    estimated_fare_inr,
    final_fare_inr,
    fare_variance_inr,

    surge_multiplier,
    CASE
        WHEN surge_multiplier = 1.00 THEN 'no_surge'
        WHEN surge_multiplier > 1.00 THEN 'surge'
        ELSE NULL
    END AS surge_band,
    CASE WHEN is_surge_trip = TRUE THEN 1 ELSE 0 END AS surge_trip_flag,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM silver_trippulse_trips_trusted;


num_affected_rows,num_inserted_rows


## 9. Validate `fact_trip`

The fact must have one row per `trip_id`, and its count and trip measure must reconcile to Trusted Silver Trips.


In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM silver_trippulse_trips_trusted) AS trusted_silver_trip_rows,
    (SELECT COUNT(*) FROM fact_trip) AS fact_trip_rows,
    (SELECT COALESCE(SUM(trip_count), 0) FROM fact_trip) AS fact_trip_count_measure,
    CASE
        WHEN (SELECT COUNT(*) FROM silver_trippulse_trips_trusted)
           = (SELECT COUNT(*) FROM fact_trip)
         AND (SELECT COUNT(*) FROM fact_trip)
           = (SELECT COALESCE(SUM(trip_count), 0) FROM fact_trip)
        THEN 'PASS'
        ELSE 'CHECK'
    END AS status;


trusted_silver_trip_rows,fact_trip_rows,fact_trip_count_measure,status
241654,241654,241654,PASS


In [0]:
%sql
SELECT trip_id, COUNT(*) AS occurrences
FROM fact_trip
GROUP BY trip_id
HAVING COUNT(*) > 1;


trip_id,occurrences


In [0]:
%sql
SELECT 'request_date_key_unresolved' AS check_name, COUNT(*) AS failures
FROM fact_trip f
LEFT JOIN dim_date d ON f.request_date_key = d.date_key
WHERE f.request_date_key IS NULL OR d.date_key IS NULL

UNION ALL

SELECT 'pickup_zone_unresolved', COUNT(*)
FROM fact_trip f
LEFT JOIN dim_zone z ON f.pickup_zone_id = z.zone_id
WHERE f.pickup_zone_id IS NULL OR z.zone_id IS NULL

UNION ALL

SELECT 'dropoff_zone_unresolved', COUNT(*)
FROM fact_trip f
LEFT JOIN dim_zone z ON f.dropoff_zone_id = z.zone_id
WHERE f.dropoff_zone_id IS NULL OR z.zone_id IS NULL

UNION ALL

SELECT 'driver_unresolved', COUNT(*)
FROM fact_trip f
LEFT JOIN dim_driver d ON f.driver_id = d.driver_id
WHERE f.driver_id IS NOT NULL AND d.driver_id IS NULL;


check_name,failures
request_date_key_unresolved,0
pickup_zone_unresolved,0
dropoff_zone_unresolved,0
driver_unresolved,0


## 10. Build `fact_payment_attempt`

**Grain:** one trusted payment attempt.

The approved payment key is `payment_id`, with `trip_id + attempt_number` also required to be unique.

No payment attempt is used to count trip requests.


In [0]:
%sql
CREATE OR REPLACE TABLE fact_payment_attempt
USING DELTA
AS
SELECT
    payment_id,
    trip_id,
    TO_DATE(payment_ts) AS payment_date_key,
    attempt_number,
    payment_ts,
    payment_method,
    payment_status,
    amount_inr,
    CASE WHEN payment_status = 'success' THEN 1 ELSE 0 END AS success_flag,
    CASE WHEN is_final_attempt = TRUE THEN 1 ELSE 0 END AS final_attempt_flag,
    payment_reference,

    1 AS attempt_count,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM silver_trippulse_payments_trusted;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM silver_trippulse_payments_trusted) AS trusted_payment_rows,
    (SELECT COUNT(*) FROM fact_payment_attempt) AS fact_payment_rows,
    (SELECT COALESCE(SUM(attempt_count), 0) FROM fact_payment_attempt) AS attempt_count_measure,
    CASE
        WHEN (SELECT COUNT(*) FROM silver_trippulse_payments_trusted)
           = (SELECT COUNT(*) FROM fact_payment_attempt)
         AND (SELECT COUNT(*) FROM fact_payment_attempt)
           = (SELECT COALESCE(SUM(attempt_count), 0) FROM fact_payment_attempt)
        THEN 'PASS'
        ELSE 'CHECK'
    END AS status;


trusted_payment_rows,fact_payment_rows,attempt_count_measure,status
172286,172286,172286,PASS


In [0]:
%sql
SELECT payment_id, COUNT(*) AS occurrences
FROM fact_payment_attempt
GROUP BY payment_id
HAVING COUNT(*) > 1;

SELECT trip_id, attempt_number, COUNT(*) AS occurrences
FROM fact_payment_attempt
GROUP BY trip_id, attempt_number
HAVING COUNT(*) > 1;


trip_id,attempt_number,occurrences


In [0]:
%sql
SELECT 'payment_date_unresolved' AS check_name, COUNT(*) AS failures
FROM fact_payment_attempt p
LEFT JOIN dim_date d ON p.payment_date_key = d.date_key
WHERE p.payment_date_key IS NULL OR d.date_key IS NULL

UNION ALL

SELECT 'payment_trip_unresolved', COUNT(*)
FROM fact_payment_attempt p
LEFT JOIN fact_trip t ON p.trip_id = t.trip_id
WHERE p.trip_id IS NULL OR t.trip_id IS NULL;


check_name,failures
payment_date_unresolved,0
payment_trip_unresolved,0


## 11. Build the five approved batch summaries

The summaries are built from the detailed Gold facts, not from Bronze or Silver Candidate.

The summary grains are kept separate so that each KPI family is evaluated at the correct business grain.


In [0]:
%sql
CREATE OR REPLACE TABLE agg_trip_operations_daily
USING DELTA
AS
SELECT
    request_date_key AS operation_date,
    service_type,
    SUM(trip_count) AS trip_requests,
    SUM(completion_flag) AS completed_trips,
    SUM(cancellation_flag) AS cancelled_trips,
    SUM(unfulfilled_flag) AS unfulfilled_trips,

    SUM(completion_flag) * 1.0 / NULLIF(SUM(trip_count), 0) AS completion_rate,
    SUM(cancellation_flag) * 1.0 / NULLIF(SUM(trip_count), 0) AS cancellation_rate,

    AVG(response_seconds) / 60.0 AS avg_response_minutes,
    AVG(trip_duration_seconds) / 60.0 AS avg_trip_duration_minutes,
    AVG(CASE WHEN completion_flag = 1 AND final_fare_inr IS NOT NULL
             THEN final_fare_inr END) AS avg_final_fare_inr,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM fact_trip
WHERE request_date_key IS NOT NULL
GROUP BY request_date_key, service_type;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE agg_zone_demand_daily
USING DELTA
AS
SELECT
    f.request_date_key AS operation_date,
    f.pickup_zone_id,
    f.service_type,

    SUM(f.trip_count) AS trip_requests,
    SUM(f.completion_flag) AS completed_trips,
    SUM(f.cancellation_flag) AS cancelled_trips,
    SUM(f.unfulfilled_flag) AS unfulfilled_trips,

    SUM(f.cancellation_flag) * 1.0 / NULLIF(SUM(f.trip_count), 0) AS cancellation_rate,
    AVG(f.response_seconds) / 60.0 AS avg_response_minutes,
    SUM(f.surge_trip_flag) * 1.0 / NULLIF(SUM(f.trip_count), 0) AS surge_trip_share,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM fact_trip f
GROUP BY f.request_date_key, f.pickup_zone_id, f.service_type;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE agg_driver_performance_daily
USING DELTA
AS
SELECT
    f.request_date_key AS operation_date,
    f.driver_id,

    SUM(f.assigned_flag) AS assigned_requests,
    SUM(f.completion_flag) AS completed_trips,
    SUM(f.driver_cancellation_flag) AS driver_cancellations,

    SUM(f.completion_flag) * 1.0 / NULLIF(SUM(f.assigned_flag), 0) AS driver_reliability_rate,
    AVG(CASE WHEN f.assigned_flag = 1 THEN f.response_seconds END) / 60.0 AS avg_response_minutes,
    AVG(CASE WHEN f.completion_flag = 1 THEN f.trip_duration_seconds END) / 60.0 AS avg_trip_duration_minutes,
    SUM(CASE WHEN f.completion_flag = 1 THEN f.final_fare_inr ELSE 0 END) AS total_final_fare_inr,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM fact_trip f
WHERE f.driver_id IS NOT NULL
GROUP BY f.request_date_key, f.driver_id;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE agg_surge_impact_daily
USING DELTA
AS
SELECT
    f.request_date_key AS operation_date,
    f.pickup_zone_id,
    f.service_type,
    f.surge_band,

    SUM(f.trip_count) AS trip_requests,
    SUM(f.surge_trip_flag) AS surge_trip_count,

    SUM(f.surge_trip_flag) * 1.0 / NULLIF(SUM(f.trip_count), 0) AS surge_trip_share,
    SUM(f.completion_flag) * 1.0 / NULLIF(SUM(f.trip_count), 0) AS completion_rate,
    SUM(f.cancellation_flag) * 1.0 / NULLIF(SUM(f.trip_count), 0) AS cancellation_rate,

    AVG(CASE WHEN f.completion_flag = 1 THEN f.final_fare_inr END) AS avg_final_fare_inr,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM fact_trip f
WHERE f.surge_band IS NOT NULL
GROUP BY
    f.request_date_key,
    f.pickup_zone_id,
    f.service_type,
    f.surge_band;


num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE agg_payment_reliability_daily
USING DELTA
AS
SELECT
    p.payment_date_key AS operation_date,
    p.payment_method,

    SUM(p.attempt_count) AS payment_attempts,
    SUM(p.success_flag) AS successful_attempts,
    SUM(CASE WHEN p.payment_status = 'failed' THEN 1 ELSE 0 END) AS failed_attempts,

    COUNT(DISTINCT CASE WHEN p.final_attempt_flag = 1
                          AND p.success_flag = 1
                        THEN p.trip_id END) AS final_successful_trips,

    SUM(p.success_flag) * 1.0 / NULLIF(SUM(p.attempt_count), 0) AS payment_success_rate,

    COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT p.trip_id), 0) AS avg_attempts_per_trip,

    SUM(CASE WHEN p.success_flag = 1 THEN p.amount_inr ELSE 0 END) AS total_success_amount_inr,

    CURRENT_TIMESTAMP() AS _gold_created_at,
    'TRIPPULSE-W07-V1' AS _gold_schema_version
FROM fact_payment_attempt p
WHERE p.payment_date_key IS NOT NULL
GROUP BY p.payment_date_key, p.payment_method;


num_affected_rows,num_inserted_rows


## 12. Validate all five summary grains

A summary must contain no duplicate rows at its declared grain.


In [0]:
%sql
SELECT 'agg_trip_operations_daily' AS gold_object, COUNT(*) AS duplicate_groups
FROM (
    SELECT operation_date, service_type
    FROM agg_trip_operations_daily
    GROUP BY operation_date, service_type
    HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'agg_zone_demand_daily', COUNT(*)
FROM (
    SELECT operation_date, pickup_zone_id, service_type
    FROM agg_zone_demand_daily
    GROUP BY operation_date, pickup_zone_id, service_type
    HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'agg_driver_performance_daily', COUNT(*)
FROM (
    SELECT operation_date, driver_id
    FROM agg_driver_performance_daily
    GROUP BY operation_date, driver_id
    HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'agg_surge_impact_daily', COUNT(*)
FROM (
    SELECT operation_date, pickup_zone_id, service_type, surge_band
    FROM agg_surge_impact_daily
    GROUP BY operation_date, pickup_zone_id, service_type, surge_band
    HAVING COUNT(*) > 1
)
UNION ALL
SELECT 'agg_payment_reliability_daily', COUNT(*)
FROM (
    SELECT operation_date, payment_method
    FROM agg_payment_reliability_daily
    GROUP BY operation_date, payment_method
    HAVING COUNT(*) > 1
);


gold_object,duplicate_groups
agg_trip_operations_daily,0
agg_zone_demand_daily,0
agg_driver_performance_daily,0
agg_surge_impact_daily,0
agg_payment_reliability_daily,0


## 13. Reconcile summaries to the detailed facts

These checks compare summary measures with the source fact at the same business scope. They do not compare summary row counts with fact row counts.


### Surge-summary reconciliation

Because every trusted Trip has a valid Week-6 surge multiplier, the two-band Gold representation must preserve the complete trip population at the date/zone/service scope.


In [0]:
%sql
WITH fact_totals AS (
    SELECT
        request_date_key AS operation_date,
        pickup_zone_id,
        service_type,
        COUNT(*) AS fact_requests
    FROM fact_trip
    GROUP BY request_date_key, pickup_zone_id, service_type
),
surge_totals AS (
    SELECT
        operation_date,
        pickup_zone_id,
        service_type,
        SUM(trip_requests) AS summary_requests
    FROM agg_surge_impact_daily
    GROUP BY operation_date, pickup_zone_id, service_type
)
SELECT
    COUNT(*) AS scope_groups,
    SUM(CASE WHEN f.fact_requests <> COALESCE(s.summary_requests, 0) THEN 1 ELSE 0 END) AS mismatched_groups,
    CASE
        WHEN SUM(CASE WHEN f.fact_requests <> COALESCE(s.summary_requests, 0) THEN 1 ELSE 0 END) = 0
        THEN 'PASS'
        ELSE 'CHECK'
    END AS surge_reconciliation
FROM fact_totals f
LEFT JOIN surge_totals s
  ON f.operation_date = s.operation_date
 AND f.pickup_zone_id = s.pickup_zone_id
 AND f.service_type = s.service_type;


scope_groups,mismatched_groups,surge_reconciliation
42945,0,PASS


### Payment final-attempt amount control

For trusted payment attempts, the approved payment summary keeps final successful trips and successful amounts visible. This check verifies that successful final attempts reconcile to the detailed payment fact.


In [0]:
%sql
WITH fact_final AS (
    SELECT
        COALESCE(SUM(CASE WHEN final_attempt_flag = 1 AND success_flag = 1 THEN 1 ELSE 0 END), 0) AS final_success_fact,
        COALESCE(SUM(CASE WHEN final_attempt_flag = 1 AND success_flag = 1 THEN amount_inr ELSE 0 END), 0) AS final_success_amount_fact
    FROM fact_payment_attempt
),
summary_final AS (
    SELECT
        COALESCE(SUM(final_successful_trips), 0) AS final_success_summary,
        COALESCE(SUM(total_success_amount_inr), 0) AS success_amount_summary
    FROM agg_payment_reliability_daily
)
SELECT *,
       CASE
           WHEN final_success_fact = final_success_summary
            AND ABS(final_success_amount_fact - success_amount_summary) < 0.01
           THEN 'PASS'
           ELSE 'CHECK'
       END AS final_payment_reconciliation
FROM fact_final
CROSS JOIN summary_final;


final_success_fact,final_success_amount_fact,final_success_summary,success_amount_summary,final_payment_reconciliation
153272,68537696.92,153272,68537696.92,PASS


In [0]:
%sql
WITH summary_total AS (
    SELECT
        (SELECT COALESCE(SUM(trip_count), 0) FROM fact_trip) AS fact_trip_total,
        (SELECT COALESCE(SUM(trip_requests), 0) FROM agg_trip_operations_daily) AS operations_total,
        (SELECT COALESCE(SUM(trip_requests), 0) FROM agg_zone_demand_daily) AS zone_total
)
SELECT *,
       CASE
           WHEN fact_trip_total = operations_total
            AND fact_trip_total = zone_total
           THEN 'PASS'
           ELSE 'CHECK'
       END AS trip_summary_reconciliation
FROM summary_total;


fact_trip_total,operations_total,zone_total,trip_summary_reconciliation
241654,241654,241654,PASS


In [0]:
%sql
WITH driver_fact AS (
    SELECT
        SUM(assigned_flag) AS assigned_fact,
        SUM(completion_flag) AS completed_fact,
        SUM(driver_cancellation_flag) AS driver_cancel_fact
    FROM fact_trip
    WHERE driver_id IS NOT NULL
),
driver_summary AS (
    SELECT
        SUM(assigned_requests) AS assigned_summary,
        SUM(completed_trips) AS completed_summary,
        SUM(driver_cancellations) AS driver_cancel_summary
    FROM agg_driver_performance_daily
)
SELECT *,
       CASE
           WHEN assigned_fact = assigned_summary
            AND completed_fact = completed_summary
            AND driver_cancel_fact = driver_cancel_summary
           THEN 'PASS'
           ELSE 'CHECK'
       END AS driver_summary_reconciliation
FROM driver_fact CROSS JOIN driver_summary;


assigned_fact,completed_fact,driver_cancel_fact,assigned_summary,completed_summary,driver_cancel_summary,driver_summary_reconciliation
212553,154613,24123,212553,154613,24123,PASS


In [0]:
%sql
WITH payment_fact AS (
    SELECT
        SUM(attempt_count) AS attempt_fact,
        SUM(success_flag) AS success_fact
    FROM fact_payment_attempt
),
payment_summary AS (
    SELECT
        SUM(payment_attempts) AS attempt_summary,
        SUM(successful_attempts) AS success_summary
    FROM agg_payment_reliability_daily
)
SELECT *,
       CASE
           WHEN attempt_fact = attempt_summary
            AND success_fact = success_summary
           THEN 'PASS'
           ELSE 'CHECK'
       END AS payment_summary_reconciliation
FROM payment_fact CROSS JOIN payment_summary;


attempt_fact,success_fact,attempt_summary,success_summary,payment_summary_reconciliation
172286,153272,172286,153272,PASS


## 14. KPI-01 to KPI-10 governed implementation

The formulas below follow the approved TripPulse KPI register.

Important control:

**Trip KPIs use `fact_trip` / trip summaries. Payment KPIs use `fact_payment_attempt` / payment summaries.**

A zero denominator returns `NULL`, not `0%`.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW trippulse_kpi_register AS
SELECT
    'KPI-01' AS kpi_id,
    'Total Trip Requests' AS kpi_name,
    'SUM(trip_count)' AS approved_formula,
    CAST(SUM(trip_count) AS DECIMAL(18,2)) AS value
FROM fact_trip

UNION ALL

SELECT
    'KPI-02',
    'Completion Rate',
    'SUM(completed_trips) / NULLIF(SUM(trip_requests), 0)',
    CAST(
        SUM(completion_flag) * 1.0 / NULLIF(SUM(trip_count), 0)
        AS DECIMAL(18,6)
    )
FROM fact_trip

UNION ALL

SELECT
    'KPI-03',
    'Cancellation Rate',
    'SUM(cancelled_trips) / NULLIF(SUM(trip_requests), 0)',
    CAST(
        SUM(cancellation_flag) * 1.0 / NULLIF(SUM(trip_count), 0)
        AS DECIMAL(18,6)
    )
FROM fact_trip

UNION ALL

SELECT
    'KPI-04',
    'Unfulfilled Rate',
    'SUM(unfulfilled_trips) / NULLIF(SUM(trip_requests), 0)',
    CAST(
        SUM(unfulfilled_flag) * 1.0 / NULLIF(SUM(trip_count), 0)
        AS DECIMAL(18,6)
    )
FROM fact_trip

UNION ALL

SELECT
    'KPI-05',
    'Average Driver Response Minutes',
    'AVG(response_seconds) / 60.0',
    CAST(AVG(response_seconds) / 60.0 AS DECIMAL(18,4))
FROM fact_trip

UNION ALL

SELECT
    'KPI-06',
    'Average Trip Duration Minutes',
    'AVG(trip_duration_seconds) / 60.0',
    CAST(AVG(trip_duration_seconds) / 60.0 AS DECIMAL(18,4))
FROM fact_trip

UNION ALL

SELECT
    'KPI-07',
    'Average Final Fare INR',
    'AVG(final_fare_inr) for completed valid fares',
    CAST(
        AVG(CASE WHEN completion_flag = 1 THEN final_fare_inr END)
        AS DECIMAL(18,2)
    )
FROM fact_trip

UNION ALL

SELECT
    'KPI-08',
    'Surge Trip Share',
    'SUM(surge_trip_count) / NULLIF(SUM(trip_requests), 0)',
    CAST(
        SUM(surge_trip_flag) * 1.0 / NULLIF(SUM(trip_count), 0)
        AS DECIMAL(18,6)
    )
FROM fact_trip

UNION ALL

SELECT
    'KPI-09',
    'Driver Reliability Rate',
    'SUM(completed_trips) / NULLIF(SUM(assigned_requests), 0)',
    CAST(
        SUM(completion_flag) * 1.0 / NULLIF(SUM(assigned_flag), 0)
        AS DECIMAL(18,6)
    )
FROM fact_trip

UNION ALL

SELECT
    'KPI-10',
    'Payment Attempt Success Rate',
    'SUM(successful_attempts) / NULLIF(SUM(payment_attempts), 0)',
    CAST(
        (SELECT SUM(success_flag) * 1.0 / NULLIF(SUM(attempt_count), 0)
         FROM fact_payment_attempt)
        AS DECIMAL(18,6)
    )
FROM fact_payment_attempt;


In [0]:
%sql
SELECT *
FROM trippulse_kpi_register
ORDER BY kpi_id;


kpi_id,kpi_name,approved_formula,value
KPI-01,Total Trip Requests,SUM(trip_count),241654.000000
KPI-02,Completion Rate,"SUM(completed_trips) / NULLIF(SUM(trip_requests), 0)",0.639811
KPI-03,Cancellation Rate,"SUM(cancelled_trips) / NULLIF(SUM(trip_requests), 0)",0.239764
KPI-04,Unfulfilled Rate,"SUM(unfulfilled_trips) / NULLIF(SUM(trip_requests), 0)",0.120424
KPI-05,Average Driver Response Minutes,AVG(response_seconds) / 60.0,3.661900
KPI-06,Average Trip Duration Minutes,AVG(trip_duration_seconds) / 60.0,47.470100
KPI-07,Average Final Fare INR,AVG(final_fare_inr) for completed valid fares,447.020000
KPI-08,Surge Trip Share,"SUM(surge_trip_count) / NULLIF(SUM(trip_requests), 0)",0.551810
KPI-09,Driver Reliability Rate,"SUM(completed_trips) / NULLIF(SUM(assigned_requests), 0)",0.727409
KPI-10,Payment Attempt Success Rate,"SUM(successful_attempts) / NULLIF(SUM(payment_attempts), 0)",0.889637


## 15. Manual KPI spot-checks

Use these simple independent checks to prove that the governed KPI output agrees with the detailed facts.

The displayed values are generated by Databricks when the notebook is executed; this notebook does not hard-code expected answers.


In [0]:
%sql
SELECT
    COUNT(*) AS trusted_trip_requests,
    SUM(completion_flag) AS completed_trips,
    SUM(cancellation_flag) AS cancelled_trips,
    SUM(unfulfilled_flag) AS unfulfilled_trips,
    AVG(response_seconds) / 60.0 AS avg_response_minutes,
    AVG(trip_duration_seconds) / 60.0 AS avg_duration_minutes,
    AVG(CASE WHEN completion_flag = 1 THEN final_fare_inr END) AS avg_final_fare_inr,
    SUM(surge_trip_flag) * 1.0 / NULLIF(COUNT(*), 0) AS surge_share,
    SUM(completion_flag) * 1.0 / NULLIF(SUM(assigned_flag), 0) AS driver_reliability
FROM fact_trip;


trusted_trip_requests,completed_trips,cancelled_trips,unfulfilled_trips,avg_response_minutes,avg_duration_minutes,avg_final_fare_inr,surge_share,driver_reliability
241654,154613,57940,29101,3.6618948372092297,47.470125625486425,447.023222,0.5518096120900130,0.7274091638320795


In [0]:
%sql
SELECT
    COUNT(*) AS payment_attempts,
    SUM(success_flag) AS successful_attempts,
    SUM(success_flag) * 1.0 / NULLIF(COUNT(*), 0) AS payment_attempt_success_rate,
    COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT trip_id), 0) AS average_attempts_per_trip
FROM fact_payment_attempt;


payment_attempts,successful_attempts,payment_attempt_success_rate,average_attempts_per_trip
172286,153272,0.8896369989436170,1.1221650491760568


## 16. Gold source-guard check

The Gold layer must not read Candidate, Bronze or Quarantine directly.

The following validation queries inspect the objects created by this notebook and confirm their row sources conceptually through the build definitions above. Reconciliation then proves their content against Trusted Silver.


In [0]:
%sql
SELECT
    'fact_trip' AS gold_object,
    'silver_trippulse_trips_trusted' AS governed_source,
    COUNT(*) AS rows
FROM fact_trip

UNION ALL

SELECT
    'fact_payment_attempt',
    'silver_trippulse_payments_trusted',
    COUNT(*)
FROM fact_payment_attempt

UNION ALL

SELECT
    'dim_zone',
    'silver_trippulse_zones_trusted',
    COUNT(*)
FROM dim_zone

UNION ALL

SELECT
    'dim_driver',
    'silver_trippulse_drivers_trusted',
    COUNT(*)
FROM dim_driver;


gold_object,governed_source,rows
fact_trip,silver_trippulse_trips_trusted,241654
fact_payment_attempt,silver_trippulse_payments_trusted,172286
dim_zone,silver_trippulse_zones_trusted,120
dim_driver,silver_trippulse_drivers_trusted,2793


## 17. Final Gold validation scorecard

All rows below should return `PASS` or zero failures before Week 7 is closed.


In [0]:
%sql
SELECT
    'dim_date_unique' AS check_name,
    CASE WHEN COUNT(*) = COUNT(DISTINCT date_key) THEN 'PASS' ELSE 'CHECK' END AS status
FROM dim_date

UNION ALL
SELECT
    'dim_zone_unique',
    CASE WHEN COUNT(*) = COUNT(DISTINCT zone_id) THEN 'PASS' ELSE 'CHECK' END
FROM dim_zone

UNION ALL
SELECT
    'dim_driver_unique',
    CASE WHEN COUNT(*) = COUNT(DISTINCT driver_id) THEN 'PASS' ELSE 'CHECK' END
FROM dim_driver

UNION ALL
SELECT
    'fact_trip_reconciles',
    CASE
        WHEN (SELECT COUNT(*) FROM fact_trip)
           = (SELECT COUNT(*) FROM silver_trippulse_trips_trusted)
        THEN 'PASS' ELSE 'CHECK'
    END

UNION ALL
SELECT
    'fact_payment_reconciles',
    CASE
        WHEN (SELECT COUNT(*) FROM fact_payment_attempt)
           = (SELECT COUNT(*) FROM silver_trippulse_payments_trusted)
        THEN 'PASS' ELSE 'CHECK'
    END

UNION ALL
SELECT
    'trip_operations_reconciles',
    CASE
        WHEN (SELECT SUM(trip_count) FROM fact_trip)
           = (SELECT SUM(trip_requests) FROM agg_trip_operations_daily)
        THEN 'PASS' ELSE 'CHECK'
    END

UNION ALL
SELECT
    'zone_demand_reconciles',
    CASE
        WHEN (SELECT SUM(trip_count) FROM fact_trip)
           = (SELECT SUM(trip_requests) FROM agg_zone_demand_daily)
        THEN 'PASS' ELSE 'CHECK'
    END

UNION ALL
SELECT
    'driver_summary_reconciles',
    CASE
        WHEN (SELECT SUM(assigned_flag) FROM fact_trip WHERE driver_id IS NOT NULL)
           = (SELECT SUM(assigned_requests) FROM agg_driver_performance_daily)
         AND (SELECT SUM(completion_flag) FROM fact_trip WHERE driver_id IS NOT NULL)
           = (SELECT SUM(completed_trips) FROM agg_driver_performance_daily)
        THEN 'PASS' ELSE 'CHECK'
    END

UNION ALL
SELECT
    'surge_summary_reconciles',
    CASE
        WHEN (
            SELECT COUNT(*)
            FROM (
                SELECT
                    f.request_date_key,
                    f.pickup_zone_id,
                    f.service_type,
                    COUNT(*) AS fact_requests
                FROM fact_trip f
                GROUP BY f.request_date_key, f.pickup_zone_id, f.service_type
            ) f
            LEFT JOIN (
                SELECT
                    operation_date,
                    pickup_zone_id,
                    service_type,
                    SUM(trip_requests) AS summary_requests
                FROM agg_surge_impact_daily
                GROUP BY operation_date, pickup_zone_id, service_type
            ) s
              ON f.request_date_key = s.operation_date
             AND f.pickup_zone_id = s.pickup_zone_id
             AND f.service_type = s.service_type
            WHERE f.fact_requests <> COALESCE(s.summary_requests, 0)
        ) = 0
        THEN 'PASS' ELSE 'CHECK'
    END

UNION ALL
SELECT
    'payment_summary_reconciles',
    CASE
        WHEN (SELECT SUM(attempt_count) FROM fact_payment_attempt)
           = (SELECT SUM(payment_attempts) FROM agg_payment_reliability_daily)
         AND (SELECT SUM(success_flag) FROM fact_payment_attempt)
           = (SELECT SUM(successful_attempts) FROM agg_payment_reliability_daily)
        THEN 'PASS' ELSE 'CHECK'
    END;


check_name,status
dim_date_unique,PASS
dim_zone_unique,PASS
dim_driver_unique,PASS
fact_trip_reconciles,PASS
fact_payment_reconciles,PASS
trip_operations_reconciles,PASS
zone_demand_reconciles,PASS
driver_summary_reconciles,PASS
surge_summary_reconciles,PASS
payment_summary_reconciles,PASS


## 18. Controlled rerun proof

For a controlled Week-7 rerun, compare the business columns of the Gold objects before and after recreating them.

Technical fields such as `_gold_created_at` may change. Business rows, keys and measures must remain stable.

### Evidence to capture

1. Gold table schemas.
2. Dimension uniqueness checks.
3. Fact reconciliation.
4. Five summary reconciliation checks.
5. KPI register.
6. One manual KPI spot-check.
7. Rerun comparison showing zero changed business rows.

Do not copy mentor/reference result values into the notebook. Execute the SQL in your own Databricks workspace.


In [0]:
%sql
SELECT
    'Week-7 completion gate' AS gate,
    CASE
        WHEN
            (SELECT COUNT(*) FROM fact_trip)
              = (SELECT COUNT(*) FROM silver_trippulse_trips_trusted)
        AND (SELECT COUNT(*) FROM fact_payment_attempt)
              = (SELECT COUNT(*) FROM silver_trippulse_payments_trusted)
        AND (SELECT SUM(trip_count) FROM fact_trip)
              = (SELECT SUM(trip_requests) FROM agg_trip_operations_daily)
        AND (SELECT SUM(trip_count) FROM fact_trip)
              = (SELECT SUM(trip_requests) FROM agg_zone_demand_daily)
        AND (SELECT SUM(attempt_count) FROM fact_payment_attempt)
              = (SELECT SUM(payment_attempts) FROM agg_payment_reliability_daily)
        THEN 'PASS'
        ELSE 'CHECK'
    END AS status;


gate,status
Week-7 completion gate,PASS


## Week 7 exit checklist

- [ ] Week-6 Candidate/Trusted/Quarantine reconciliation is PASS.
- [ ] `dim_date` created and unique.
- [ ] `dim_zone` contains accepted zones only.
- [ ] `dim_driver` contains accepted drivers only.
- [ ] `fact_trip` is one row per trusted `trip_id`.
- [ ] `fact_payment_attempt` is one row per trusted `payment_id`.
- [ ] `trip_id + attempt_number` is unique for payment attempts.
- [ ] Five approved batch summaries are created at their declared grains.
- [ ] Fact and summary totals reconcile.
- [ ] KPI-01 through KPI-10 are implemented with approved denominator handling.
- [ ] Zero denominators return NULL rather than false 0%.
- [ ] No Gold object reads Bronze, Silver Candidate or Quarantine.
- [ ] Actual Databricks outputs are captured as Week-7 evidence.
- [ ] GitHub notebook and weekly log are updated.

**Week-7 boundary:** Gold model, KPI implementation and reconciliation. Power BI work begins in Week 8.


## Viva points

**Why separate the two facts?**  
A trip is one request, while a payment can have multiple attempts. Joining them directly can multiply trip rows.

**Why is `trip_count = 1` in `fact_trip`?**  
It makes the trip-request grain explicit and lets summaries use `SUM(trip_count)`.

**Why is `payment_attempt` not used for Total Trip Requests?**  
Because payment attempts are a different grain and retries would overcount requests.

**Why use Trusted Silver only?**  
Gold is the decision-ready layer and must not re-introduce Candidate or quarantined records.

**Why return NULL for a zero denominator?**  
A zero denominator means the rate is undefined; displaying 0% would falsely imply a valid population with zero success.
